# BB84 on real hardware: telling noise apart from an eavesdropper

*Part of the QUEST Cryptography & Security series*

BB84 is normally taught as a protocol that detects eavesdropping. Eve measures the qubits in transit, her measurements disturb them, and Alice and Bob catch her by comparing part of their sifted key. The standard demonstration runs on an ideal simulator, reports a quantum bit error rate (QBER) of zero without Eve and 25% with her, and stops there.

Real hardware has no zero. Gate error, decoherence and readout error all produce the same signature as an eavesdropper, and the protocol has no way to tell them apart. Security therefore requires assuming the worst: every observed error is attributed to Eve, and key material is discarded to compensate. This notebook measures what that costs on current devices.

We implement BB84 end to end, add an intercept-resend eavesdropper, derive the 11% abort threshold instead of quoting it, then run the protocol on three real processors with no eavesdropper present. Sweeping the length of the quantum channel shows where each device stops being able to carry the protocol at all.

**Learning objectives.** By the end of this notebook you will:

1. Implement BB84 including basis reconciliation, sifting, and QBER estimation.
2. Model an intercept-resend eavesdropper and predict the QBER she induces.
3. Derive the 11% security threshold from the Shor-Preskill key rate rather than taking it as given.
4. Measure the intrinsic QBER of real quantum processors running the protocol with no eavesdropper.
5. Convert a measured QBER into a secure key rate and quantify what hardware noise costs.
6. Explain why deployed QKD uses photonic links rather than gate-based processors.

**What to bring in.** Single-qubit states and measurement in the $Z$ and $X$ bases. No cryptography background is assumed; the protocol is built from scratch. The security argument is sketched rather than proved.

**Credit budget.** Simulator work is free. Hardware runs use 1,000 shots per channel length, 6 channel lengths, 3 backends, for 18,000 shots total. At typical rates this is a few dollars in credits.

## The protocol

Alice holds a random bit $a$ and a random basis choice. She encodes $a$ in one of two bases:

| Basis | $a = 0$ | $a = 1$ |
|---|---|---|
| $Z$ (rectilinear) | $\lvert 0 \rangle$ | $\lvert 1 \rangle$ |
| $X$ (diagonal) | $\lvert + \rangle$ | $\lvert - \rangle$ |

She sends the qubit to Bob, who measures it in a basis he picks at random, independently of Alice. Afterwards the two announce their basis choices on a public classical channel, never their bits, and discard every round where the bases disagree. This step is **sifting** and it keeps about half the rounds.

In the surviving rounds Bob measured in the basis Alice prepared in, so with a noiseless channel his bit equals hers. Any disagreement is an error, and the **quantum bit error rate** is

$$Q = \frac{\#\{i \in \text{sifted} : b_i \neq a_i\}}{\#\{\text{sifted}\}}$$

Alice and Bob estimate $Q$ by publicly comparing a random subset of their sifted bits, then discard the bits they compared. The security claim of BB84 is that $Q$ bounds how much Eve can know about the rest.

The security rests on two facts about the four states. They span two mutually unbiased bases, so measuring a $Z$-basis state in the $X$ basis returns a uniformly random outcome and destroys the encoded bit. And no measurement distinguishes all four, so Eve cannot read a qubit without choosing a basis and risking that she chose wrong.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import brentq

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

sim = AerSimulator(seed_simulator=42)

print("Setup complete.")

## One round of the protocol

Each round uses a single qubit. Alice's preparation is an optional $X$ gate to set the bit, then an optional $H$ to move into the $X$ basis. Bob's measurement is an optional $H$ to rotate his chosen basis onto the computational basis, then a computational-basis measurement.

We write `0` for the $Z$ basis and `1` for the $X$ basis throughout, and we keep Qiskit's little-endian convention: `qc.qubits[0]` is the rightmost bit of a returned bitstring.

The `channel_reps` argument inserts pairs of $X$ gates to represent time spent in transit. Each pair is logically the identity, so the ideal outcome is unchanged, but on hardware every gate carries error. It gives us a knob that lengthens the channel without changing what the protocol should return.

In [ ]:
Z_BASIS, X_BASIS = 0, 1


def bb84_round(alice_bit, alice_basis, bob_basis, channel_reps=0):
    """One BB84 round on a single qubit."""
    qc = QuantumCircuit(1, 1)

    # Alice prepares
    if alice_bit:
        qc.x(0)
    if alice_basis == X_BASIS:
        qc.h(0)
    qc.barrier()

    # The channel: channel_reps identity-equivalent X pairs
    for _ in range(channel_reps):
        qc.x(0)
        qc.barrier()
        qc.x(0)
        qc.barrier()

    # Bob measures in his chosen basis
    if bob_basis == X_BASIS:
        qc.h(0)
    qc.measure(0, 0)
    return qc


bb84_round(alice_bit=1, alice_basis=X_BASIS, bob_basis=X_BASIS, channel_reps=1).draw('mpl')

The barriers are load-bearing. Without them the transpiler recognises that $X \cdot X = I$ and deletes the channel, which is correct logically and useless physically. Barriers block that optimisation so the gates survive to the device. The compilation notebook in the Systems & Hardware series looks at when you want the compiler to do this and when you do not.

## Running the protocol without an eavesdropper

Alice generates random bits and bases, Bob generates random bases, and we run every round. Eve is built into the same function so we can switch her on later: she intercepts a fraction `eve_fraction` of the rounds, measures each in a basis she chooses at random, and forwards a fresh qubit prepared from her outcome in her own basis.

In [ ]:
def run_protocol(n_rounds, channel_reps=0, eve_fraction=0.0, backend=None, seed=None):
    """
    Run n_rounds of BB84 and return the sifted keys and the QBER.

    eve_fraction is the fraction of rounds an intercept-resend eavesdropper touches.
    backend defaults to the ideal Aer simulator.
    """
    rng = np.random.default_rng(seed)
    backend = sim if backend is None else backend

    alice_bits = rng.integers(0, 2, n_rounds)
    alice_bases = rng.integers(0, 2, n_rounds)
    bob_bases = rng.integers(0, 2, n_rounds)

    # Eve intercepts a random subset of rounds, measuring in a random basis
    intercepted = rng.random(n_rounds) < eve_fraction
    eve_bases = rng.integers(0, 2, n_rounds)

    sent_bits, sent_bases = alice_bits.copy(), alice_bases.copy()
    for i in np.flatnonzero(intercepted):
        if eve_bases[i] == alice_bases[i]:
            continue                          # right basis: she learns the bit, state passes intact
        sent_bits[i] = rng.integers(0, 2)     # wrong basis: her outcome is uniformly random
        sent_bases[i] = eve_bases[i]          # and she resends in her own basis

    circuits = [bb84_round(sent_bits[i], sent_bases[i], bob_bases[i], channel_reps)
                for i in range(n_rounds)]
    counts = backend.run(circuits, shots=1).result().get_counts()
    bob_bits = np.array([int(next(iter(c))) for c in counts])

    sifted = alice_bases == bob_bases
    qber = float((bob_bits[sifted] != alice_bits[sifted]).mean())
    return {'alice_key': alice_bits[sifted], 'bob_key': bob_bits[sifted],
            'n_sifted': int(sifted.sum()), 'qber': qber}

In [ ]:
N_ROUNDS = 4000

clean = run_protocol(N_ROUNDS, seed=1)
alice_str = ''.join(map(str, clean['alice_key'][:40]))
bob_str = ''.join(map(str, clean['bob_key'][:40]))

print(f"Rounds run:        {N_ROUNDS}")
print(f"Sifted key length: {clean['n_sifted']}  ({clean['n_sifted'] / N_ROUNDS:.1%} of rounds)")
print(f"QBER:              {clean['qber']:.4f}")
print(f"Alice, first 40 sifted bits: {alice_str}")
print(f"Bob,   first 40 sifted bits: {bob_str}")

Sifting keeps close to half the rounds, which is what two independent fair coins agreeing should give. On the ideal simulator the QBER is exactly zero and the two sifted keys are identical. Everything from here on is a departure from this baseline.

## Adding an eavesdropper

Intercept-resend is the simplest attack. Eve captures the qubit, measures it in a basis she picks at random, and sends Bob a fresh qubit prepared according to her outcome, in her basis.

Half the time she picks Alice's basis. Her measurement is then deterministic, she learns the bit, and the state she forwards is the one Alice sent. She is invisible in those rounds.

The other half she picks the conjugate basis. Her outcome is uniformly random and tells her nothing about Alice's bit, and the state she forwards is in the wrong basis. When such a round survives sifting, Bob measures a conjugate-basis state and gets a random result, disagreeing with Alice half the time.

So a fully intercepting Eve induces

$$Q_{\text{Eve}} = \underbrace{\tfrac{1}{2}}_{\text{wrong basis}} \times \underbrace{\tfrac{1}{2}}_{\text{random outcome}} = \tfrac{1}{4}$$

and intercepting a fraction $f$ of rounds gives $Q = f/4$. That linearity is what makes the QBER useful: it reads out how much of the traffic Eve touched.

In [ ]:
fractions = np.linspace(0, 1, 11)
eve_qbers = [run_protocol(3000, eve_fraction=f, seed=100 + i)['qber']
             for i, f in enumerate(fractions)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fractions, fractions / 4, 'k--', linewidth=2, alpha=0.6, label='Prediction $Q = f/4$')
ax.plot(fractions, eve_qbers, 'o-', color='#a02580', markersize=9, linewidth=2,
        markeredgecolor='white', markeredgewidth=1.2, label='Simulated')
ax.set_xlabel('Fraction of rounds Eve intercepts, $f$')
ax.set_ylabel('QBER')
ax.set_ylim(0, 0.3)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"QBER at f = 1: {eve_qbers[-1]:.4f}  (predicted 0.2500)")

## Where the 11% comes from

Courses usually state that BB84 aborts above a QBER of roughly 11%. The number is worth deriving, because it is what turns hardware noise from an annoyance into a cost.

After sifting, Alice and Bob hold correlated but unequal strings, and Eve holds partial information about them. They run error correction to reconcile the strings, which leaks more information to Eve, then privacy amplification to compress the result into a shorter string about which Eve knows essentially nothing. For BB84 with one-way post-processing, the Shor-Preskill analysis gives an asymptotic secure key rate per sifted bit of

$$r(Q) = 1 - 2h(Q), \qquad h(Q) = -Q\log_2 Q - (1 - Q)\log_2 (1 - Q)$$

One factor of $h(Q)$ pays for error correction, the other bounds Eve's information. The rate reaches zero when $h(Q) = 1/2$, which has no closed form, so we solve it numerically.

In [ ]:
def binary_entropy(q):
    """Binary entropy in bits, with h(0) = h(1) = 0."""
    q = np.atleast_1d(np.asarray(q, dtype=float))
    out = np.zeros_like(q)
    m = (q > 0) & (q < 1)
    out[m] = -q[m] * np.log2(q[m]) - (1 - q[m]) * np.log2(1 - q[m])
    return out


def key_rate(q):
    """Asymptotic Shor-Preskill secure key rate per sifted bit."""
    return np.maximum(0.0, 1 - 2 * binary_entropy(q))


q_threshold = brentq(lambda q: 1 - 2 * binary_entropy(q)[0], 1e-9, 0.5 - 1e-9)

print(f"Zero-rate threshold:              Q* = {q_threshold:.4%}")
print(f"Eve interception fraction at Q*:   f = {4 * q_threshold:.3f}")
for q in [0.01, 0.02, 0.05, 0.08]:
    print(f"  Q = {q:.0%}  ->  r = {key_rate(q)[0]:.3f}")

In [ ]:
q_grid = np.linspace(0, 0.2, 400)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(q_grid, key_rate(q_grid), color='#1a5285', linewidth=2.5)
ax.fill_between(q_grid, 0, key_rate(q_grid), color='#1a5285', alpha=0.12)
ax.axvline(q_threshold, color='k', linestyle='--', alpha=0.6)
ax.annotate(f'$Q^*$ = {q_threshold:.2%}',
            xy=(q_threshold, 0.0), xytext=(q_threshold + 0.012, 0.28), fontsize=11,
            arrowprops=dict(arrowstyle='->', alpha=0.6))
ax.set_xlabel('QBER')
ax.set_ylabel('Secure key rate per sifted bit, $r$')
ax.set_title('Shor-Preskill key rate for BB84')
ax.set_xlim(0, 0.2)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Two things to take from this curve.

The threshold sits at $Q^* = 11.00\%$, and since $Q = f/4$ that corresponds to Eve intercepting 44% of the rounds. She can tap nearly half the traffic before the abort rule fires. The protocol is not detecting her presence, it is bounding her knowledge, and it keeps producing key as long as privacy amplification can outrun her.

The rate also falls steeply well before the threshold. At $Q = 2\%$ it is already down to about $0.72$, and at $Q = 5\%$ to about $0.43$. Errors are expensive long before they are fatal, which matters because a real device produces them whether or not anyone is listening.

## Restructuring the protocol for hardware

The implementation above submits one circuit per round. On hardware that would mean thousands of separate jobs, which is neither affordable nor necessary.

Sifting and basis reconciliation are classical bookkeeping and need no quantum device. The only quantity the device sets is the probability that Bob's outcome differs from Alice's bit in a round that survived sifting, and that probability depends on which of the four states Alice sent, not on the random basis draws. So we prepare all four BB84 states in parallel on four qubits, measure each in its matching basis, and read the error rate off the shot record. One circuit then gives `shots` rounds' worth of statistics for every state at once.

In [ ]:
BB84_STATES = [(0, Z_BASIS), (1, Z_BASIS), (0, X_BASIS), (1, X_BASIS)]
STATE_LABELS = ['|0>', '|1>', '|+>', '|->']


def sifted_error_circuit(channel_reps=0):
    """Prepare all four BB84 states in parallel, one per qubit, each measured in its own basis."""
    n = len(BB84_STATES)
    qc = QuantumCircuit(n, n)

    for q, (bit, basis) in enumerate(BB84_STATES):
        if bit:
            qc.x(q)
        if basis == X_BASIS:
            qc.h(q)
    qc.barrier()

    for _ in range(channel_reps):
        qc.x(range(n))
        qc.barrier()
        qc.x(range(n))
        qc.barrier()

    for q, (bit, basis) in enumerate(BB84_STATES):
        if basis == X_BASIS:
            qc.h(q)
    qc.measure(range(n), range(n))
    return qc


def per_state_error_rates(counts, shots):
    """
    Error rate for each of the four BB84 states from a packed-circuit shot record.
    Qiskit puts the highest-indexed clbit leftmost, so clbit q is bitstring[n - 1 - q].
    """
    n = len(BB84_STATES)
    errors = np.zeros(n)
    for bitstring, num in counts.items():
        b = bitstring.replace(' ', '')
        for q, (bit, _) in enumerate(BB84_STATES):
            if int(b[n - 1 - q]) != bit:
                errors[q] += num
    return errors / shots


def qber_from_counts(counts, shots):
    """QBER averaged over the four BB84 states."""
    return float(per_state_error_rates(counts, shots).mean())


sifted_error_circuit(channel_reps=1).draw('mpl', fold=90)

In [ ]:
# Confirm the packed circuit reproduces the ideal result before spending credits
for reps in [0, 4, 16]:
    counts = sim.run(sifted_error_circuit(reps), shots=4000).result().get_counts()
    rates = per_state_error_rates(counts, 4000)
    print(f"channel_reps = {reps:2d}   ideal QBER = {rates.mean():.4f}   per state = {rates}")

The ideal QBER stays at zero for every channel length, which is the point of building the channel out of identity pairs. Anything a real device reports from here is hardware, not protocol.

## Running on three processors

We use the same three devices as the Grover notebook in the Foundations & Algorithms series, spanning both modalities in wide use. BB84 needs only single-qubit gates and measurement, so connectivity is irrelevant here. What matters is single-qubit gate fidelity and readout fidelity.

| Backend | Vendor | Modality | Relevant error sources |
|---|---|---|---|
| `rigetti_ankaa_3` | Rigetti | Superconducting | Single-qubit gate error, readout error, $T_1/T_2$ |
| `iqm_garnet` | IQM | Superconducting | Single-qubit gate error, readout error, $T_1/T_2$ |
| `aqt_marmot` | AQT | Trapped ion | Single-qubit gate error, readout error, slower gates |

*Device names are illustrative. Check `provider.get_devices()` for what is currently available in your account and substitute accordingly.*

**Note on queue times.** This is 18 separate hardware jobs. Depending on device load the submission cell may take minutes to hours.

In [ ]:
provider = QbraidProvider()

# Instructor: update these IDs to match currently available hardware in your account.
BACKENDS = {
    'Rigetti Ankaa-3': 'rigetti_ankaa_3',
    'IQM Garnet':      'iqm_garnet',
    'AQT Marmot':      'aqt_marmot',
}

COLORS = {
    'Rigetti Ankaa-3': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT Marmot':      '#2d7a4f',
}

CHANNEL_REPS = [0, 1, 2, 4, 8, 16]
SHOTS = 1000

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}
print("Configured backends:", list(devices))

In [ ]:
hw_qber = {name: {} for name in BACKENDS}
hw_per_state = {name: {} for name in BACKENDS}

for name, device in devices.items():
    for reps in CHANNEL_REPS:
        qc = sifted_error_circuit(reps)
        job = device.run(qc, shots=SHOTS)
        counts = job.result().data.get_counts()
        hw_qber[name][reps] = qber_from_counts(counts, SHOTS)
        hw_per_state[name][reps] = per_state_error_rates(counts, SHOTS)
        print(f"{name:18s} channel_reps={reps:2d}  QBER = {hw_qber[name][reps]:.4f}")

## QBER against channel length

The ideal curve is flat at zero. Every device sits above it, with no eavesdropper anywhere in the experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6))

ax.axhline(0, color='k', linestyle='--', linewidth=2, alpha=0.5, label='Ideal (no Eve)')
ax.axhline(q_threshold, color='#8a1c1c', linestyle='-.', linewidth=2,
           label=f'Abort threshold $Q^*$ = {q_threshold:.1%}')
ax.axhline(0.25, color='gray', linestyle=':', linewidth=2,
           label='Fully intercepting Eve (25%)')

for name in BACKENDS:
    reps = sorted(hw_qber[name])
    ax.plot(reps, [hw_qber[name][r] for r in reps], 'o-', color=COLORS[name], label=name,
            markersize=11, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Channel length (identity-pair repetitions)')
ax.set_ylabel('QBER')
ax.set_title('BB84 error rate on real hardware, with no eavesdropper present')
ax.set_ylim(-0.01, 0.3)
ax.grid(alpha=0.3)
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

At `channel_reps = 0` the circuit is a state preparation followed immediately by a measurement, so the QBER there is essentially readout error plus a gate or two. Expect a few percent on a good device. That already costs real key: at $Q = 3\%$ the rate is about $0.61$, so nearly 40% of the sifted key is spent defending against an eavesdropper who does not exist.

As the channel lengthens the QBER climbs and eventually crosses $Q^*$. Past that point the protocol yields no secure key at all, and Alice and Bob are obliged to abort even though the only thing in the channel is the device's own noise.

## What the noise costs

Convert each measured QBER into a key rate. The last columns are the quantities an operator would actually care about: how many secure bits survive, and what level of eavesdropping the observed error rate is indistinguishable from.

In [ ]:
rows = []
for name in BACKENDS:
    for reps in sorted(hw_qber[name]):
        q = hw_qber[name][reps]
        r = float(key_rate(q)[0])
        rows.append({
            'Backend': name,
            'Channel reps': reps,
            'QBER': f'{q:.4f}',
            'Key rate r(Q)': f'{r:.3f}',
            'Secure bits per 1000 sifted': f'{1000 * r:.0f}',
            'Equivalent Eve fraction': f'{min(4 * q, 1.0):.2f}',
            'Verdict': 'ABORT' if q >= q_threshold else 'ok',
        })

pd.DataFrame(rows).set_index(['Backend', 'Channel reps'])

The "equivalent Eve fraction" column is the honest reading of each row. A device showing a QBER of 6% is indistinguishable, from inside the protocol, from a noiseless device with an eavesdropper intercepting a quarter of the traffic. Alice and Bob cannot open the box and check which one they have, so they are required to assume the second.

## Which states are noisiest

The four states are not equally robust. $\lvert 0 \rangle$ needs no gates at all before measurement, $\lvert 1 \rangle$ needs one $X$, and the two $X$-basis states need an $H$ at each end. On top of that, readout error on superconducting devices is usually asymmetric, because a qubit in $\lvert 1 \rangle$ can decay toward $\lvert 0 \rangle$ during the measurement itself but not the other way round.

In [ ]:
per_state_rows = []
for name in BACKENDS:
    rates = hw_per_state[name][0]
    row = {'Backend': name}
    row.update({lbl: f'{rate:.4f}' for lbl, rate in zip(STATE_LABELS, rates)})
    row['Mean'] = f'{rates.mean():.4f}'
    per_state_rows.append(row)

pd.DataFrame(per_state_rows).set_index('Backend')

If the $\lvert 1 \rangle$ column is consistently worse than $\lvert 0 \rangle$, that is asymmetric readout error, and it is largely correctable in classical post-processing. If the two $X$-basis columns are worse than both $Z$-basis columns, that is the cost of the extra $H$ gates. Separating the two tells you whether to invest in readout calibration or in gate calibration. The device-selection notebook in the Systems & Hardware series builds this diagnosis out properly.

## Why deployed QKD does not look like this

Nothing above is a QKD system, and it is worth being precise about why.

The qubit never leaves the chip. In a real link the quantum state crosses a physical distance, which is what gives Eve something to intercept. Here Alice, Bob and Eve are all the same processor and the channel is a few gates on one qubit. What we implemented is the protocol's logic and its error accounting, not its threat model.

Deployed QKD uses photons in fibre or free space, where the dominant impairment is loss rather than gate error. A photon that never arrives is not an error, it is a missing round, and it does not raise the QBER. That distinction is central: BB84 tolerates enormous loss and very little error, and the two are handled by different parts of the analysis.

Real systems also face attacks with no analogue on a gate-based processor. A laser pulse attenuated to one photon on average sometimes contains two, and Eve can keep one and forward the other, learning a bit while disturbing nothing; decoy-state protocols exist to bound exactly this. Attacks on Bob's detectors rather than on the channel motivated measurement-device-independent QKD.

What the experiment does show, quantitatively, is the equivalence at the centre of the security argument. The protocol has one observable, $Q$, and it cannot ask where the errors came from. Every mechanism that produces them is charged to Eve. On current processors that charge consumes a large fraction of the key at zero channel length, and all of it not long after.

## Where to go next

- **Estimate $Q$ from a sample rather than from the whole key.** Alice and Bob compare a random subset and discard it. Sample 10% of the sifted bits, estimate $Q$ from those, and put a confidence interval on it. How many rounds are needed before the estimate is tight enough to trust a decision near the threshold?
- **Add finite-key effects.** The rate $r = 1 - 2h(Q)$ is asymptotic. Real systems run finite blocks and pay a penalty that grows as blocks shrink. Compare the asymptotic rate against a finite-key bound at block sizes of $10^4$ and $10^6$.
- **Let the compiler eat the channel.** Rerun the hardware cell with the barriers removed and `optimization_level=3`. The QBER should collapse to the `channel_reps = 0` value at every length, because the compiler removed the channel entirely. Worth seeing once.
- **Build E91 instead.** Distribute entangled pairs, measure in randomly chosen bases, and use a CHSH violation rather than a QBER as the security witness. Compare what the two protocols demand of the hardware.
- **Reconcile the keys you actually measured.** Implement Cascade on the sifted keys from the hardware run and check how many bits of leakage it costs against the $h(Q)$ bound.

---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which cryptography topic would you most want to see analyzed on real hardware next?

Please submit your responses via the QUEST portal or reply to your instructor.